In [12]:
import mlx.core as mx
from mlx_lm import load
import pandas as pd

messages = [
    {
        "role": "user",
        "content": "Subtract 2 from 3 in Python.",
    },
    {
        "role": "assistant",
        "content": "```python\n2 - 3\n```",
    },
    {
        "role": "user",
        "content": "Wrong! The answer should be 1.",
    }
]

model, tokenizer = load("mlx-community/Qwen2.5-Coder-7B-Instruct-4bit")

ids = mx.array(tokenizer.apply_chat_template(messages))
hidden = model.model(mx.concat([ids, ids])[None])[0,len(ids):]
logits = model.lm_head(hidden)
probs = mx.softmax(logits, axis=-1)
labels = ids[1:]
pred_ids = mx.argsort(probs, axis=-1)[:,:-(3+1):-1]
mx.eval(pred_ids)

rows = []
for i in range(len(labels)):
    def cell(id):
        token = tokenizer.decode([id.tolist()], skip_special_tokens=False)
        return f"{token!r} @ {probs[i,id]:.3f}"
    
    row = {"label": cell(labels[i])}
    for j, pred_id in enumerate(pred_ids[i]):
        row[f"pred_{j}"] = cell(pred_id)
    rows.append(row)
df = pd.DataFrame(rows)
df.tail(60)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

,label,pred_0,pred_1,pred_2
3,' are' @ 0.572,' are' @ 0.572,"""'re"" @ 0.425",' can' @ 0.000
4,' Q' @ 0.000,' correct' @ 0.862,' right' @ 0.100,' absolutely' @ 0.016
5,'wen' @ 1.000,'wen' @ 1.000,'W' @ 0.000,' Wen' @ 0.000
6,"',' @ 0.995","',' @ 0.995",'.' @ 0.002,' and' @ 0.001
7,' created' @ 0.220,' an' @ 0.445,' created' @ 0.220,' a' @ 0.210
8,' by' @ 1.000,' by' @ 1.000,' to' @ 0.000,' and' @ 0.000
9,' Alibaba' @ 1.000,' Alibaba' @ 1.000,' Ali' @ 0.000,'阿里' @ 0.000
10,' Cloud' @ 1.000,' Cloud' @ 1.000,'Cloud' @ 0.000,' cloud' @ 0.000
11,'.' @ 0.968,'.' @ 0.968,"',' @ 0.022",' and' @ 0.008
12,' You' @ 0.095,' I' @ 0.745,' You' @ 0.095,' Let' @ 0.027


There's credit attribution, but it falls off after the first instance because the model's disagreement isn't reflected in the trajectory and so it tries to conform with what's already there, even if it's incorrect. I think this means we can't do a naive offline approach. Maybe we can try chunks?